# Stateful Typed Memory-Augmented Architecture
### Inspired by LongNAP (Shaikh et al. 2026) + MIRIX Memory Taxonomy

## What this is

A stateful memory-augmented inference system where **Qwen autonomously decides**
what to store, what type it is, and what to retrieve — without modifying base model weights.

```
For each dialogue turn:

  [RETRIEVE]
  Search all three memory banks with current turn as query
  → factual_memory.retrieve(query)
  → preference_memory.retrieve(query)  
  → behavioral_memory.retrieve(query)

  [QWEN READS TURN + RETRIEVED MEMORY]
  → Understands what was just said
  → Decides what new information to store
  → Outputs structured memory operations:
     STORE_FACT: Caroline is pursuing adoption
     STORE_PREFERENCE: Caroline values direct feedback
     STORE_BEHAVIOR: Caroline seeks validation before decisions
     NO_STORE: nothing new here

  [UPDATE MEMORY]
  → Execute Qwen's memory operations
  → Save to disk (persistent across runs)

[WHEN QA PAIR ARRIVES]
  → Retrieve from all three memory banks
  → Qwen reasons about what is needed
  → GPT-4o-mini answers
  → Judge scores
```

## How this differs from session-by-session pipeline

| | Session pipeline | This pipeline |
|---|---|---|
| Memory structure | Flat single bank | Three typed banks |
| Memory controller | Python (fixed rules) | Qwen (autonomous decisions) |
| Granularity | Per session | Per turn |
| Session summaries | Yes | No — Qwen extracts specific facts |
| Retrieval | One BM25 search | Three typed searches |
| Persistence | RAM only | Disk (survives restarts) |

## Models
- **Qwen-2.5-7B-Instruct** — memory controller + reasoning (frozen)
- **GPT-4o-mini** — answering + judging


## 1. Install

In [1]:
!pip install -q transformers accelerate rank_bm25 openai tqdm
print("Done.")

Done.


## 2. Imports & Config

In [ ]:
import json, os, re, time, warnings
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI

warnings.filterwarnings("ignore")

# ── API Key ──
OPENAI_API_KEY = ""
# from google.colab import userdata; OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# ── Models ──
QWEN_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
OPENAI_MODEL = "gpt-4o-mini"

# ── Memory config ──
TOP_K_PER_BANK = 3      # retrieve top-3 from each bank = 9 total
MEMORY_DIR     = Path("./memory_store")  # disk persistence directory

# ── Evaluation config ──
MAX_TURNS_TO_PROCESS = None  # set to int to cap (None = all turns)
MAX_QA_TO_ANSWER     = 30    # cap QA pairs for quick testing
DEBUG = True                  # show every step

# ── Category map ──
CATEGORY_MAP = {
    1: "single-hop", 2: "multi-hop",
    3: "temporal",   4: "open-domain", 5: "adversarial",
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Device: cuda
GPU:  NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 3. Load Qwen-2.5-7B-Instruct

In [3]:
print(f"Loading {QWEN_MODEL} in bfloat16...")
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Loaded. {sum(p.numel() for p in model.parameters())/1e9:.1f}B params")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


def qwen_generate(prompt: str, max_new_tokens: int = 400) -> str:
    """
    Stateless Qwen call.
    The model has no memory — all state lives in Python objects.
    We inject relevant memory into each prompt explicitly.
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print("\nTest:", qwen_generate("Say hello in one sentence."))

Loading Qwen/Qwen2.5-7B-Instruct in bfloat16...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded. 7.6B params
VRAM: 15.2 GB

Test: Hello there!


## 4. Typed Memory Banks
Three separate banks. Each searched independently. Each stores different information type.
Persisted to disk so memory survives across notebook restarts.


In [4]:
@dataclass
class MemoryEntry:
    memory_type : str    # "factual" | "preference" | "behavioral"
    content     : str    # the actual stored text
    source_dia  : str    # which dialogue turn this came from
    session_num : int    # which session
    timestamp   : str    # session timestamp
    entry_id    : int


class TypedMemoryBank:
    """
    Single typed memory bank (factual OR preference OR behavioral).
    BM25 retrieval over stored entries.
    Persisted to disk as JSON.
    """
    def __init__(self, memory_type: str, save_path: Path):
        self.memory_type = memory_type
        self.save_path   = save_path
        self.entries     : list[MemoryEntry] = []
        self.bm25        = None
        self._next_id    = 0
        self._load()

    def _load(self):
        """Load from disk if exists."""
        if self.save_path.exists():
            with open(self.save_path) as f:
                data = json.load(f)
            self.entries  = [MemoryEntry(**e) for e in data["entries"]]
            self._next_id = data.get("next_id", len(self.entries))
            self._rebuild_index()
            print(f"  Loaded {len(self.entries)} {self.memory_type} memories from disk")

    def _save(self):
        """Persist to disk."""
        self.save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(self.save_path, "w") as f:
            json.dump({
                "memory_type": self.memory_type,
                "next_id"    : self._next_id,
                "entries"    : [vars(e) for e in self.entries],
            }, f, indent=2)

    def _rebuild_index(self):
        if self.entries:
            tokenized = [e.content.lower().split() for e in self.entries]
            self.bm25 = BM25Okapi(tokenized)

    def add(self, content: str, source_dia: str,
            session_num: int, timestamp: str) -> MemoryEntry:
        """Add entry and persist to disk."""
        self._next_id += 1
        entry = MemoryEntry(
            memory_type = self.memory_type,
            content     = content,
            source_dia  = source_dia,
            session_num = session_num,
            timestamp   = timestamp,
            entry_id    = self._next_id,
        )
        self.entries.append(entry)
        self._rebuild_index()
        self._save()
        return entry

    def retrieve(self, query: str, top_k: int = TOP_K_PER_BANK) -> list[MemoryEntry]:
        """BM25 retrieval over this bank's entries."""
        if not self.entries or self.bm25 is None:
            return []
        scores  = self.bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [self.entries[i] for i in top_idx if scores[i] > 0]

    def __len__(self):
        return len(self.entries)

    def summary(self) -> str:
        return f"{len(self.entries)} {self.memory_type} entries"


class MemorySystem:
    """
    Three typed memory banks together.
    Qwen is the controller — decides what goes where.
    """
    def __init__(self, user_id: str, base_dir: Path = MEMORY_DIR):
        user_dir = base_dir / user_id
        self.factual    = TypedMemoryBank("factual",    user_dir / "factual.json")
        self.preference = TypedMemoryBank("preference", user_dir / "preference.json")
        self.behavioral = TypedMemoryBank("behavioral", user_dir / "behavioral.json")
        self.user_id    = user_id

    def retrieve_all(self, query: str, top_k: int = TOP_K_PER_BANK) -> dict:
        """Retrieve from all three banks and return separately."""
        return {
            "factual"   : self.factual.retrieve(query,    top_k),
            "preference": self.preference.retrieve(query, top_k),
            "behavioral": self.behavioral.retrieve(query, top_k),
        }

    def format_for_context(self, retrieved: dict) -> str:
        """Format retrieved entries from all banks for prompt injection."""
        parts = []
        for bank_name, entries in retrieved.items():
            if entries:
                bank_lines = [f"[{bank_name.upper()} MEMORY]"]
                for e in entries:
                    bank_lines.append(f"  - {e.content} (from {e.source_dia})")
                parts.append("\n".join(bank_lines))
        return "\n\n".join(parts) if parts else "No relevant memory available."

    def summary(self) -> str:
        return (f"factual={len(self.factual)} "
                f"preference={len(self.preference)} "
                f"behavioral={len(self.behavioral)}")

    def total(self) -> int:
        return len(self.factual) + len(self.preference) + len(self.behavioral)


# ── test ──
MEMORY_DIR.mkdir(parents=True, exist_ok=True)
test_mem = MemorySystem("test_user")
print(f"Memory system initialized: {test_mem.summary()}")

Memory system initialized: factual=0 preference=0 behavioral=0


## 5. Load LoCoMo Dataset

In [5]:
!git clone -q https://github.com/snap-research/locomo
DATA_PATH = "locomo/data/locomo10.json"

with open(DATA_PATH) as f:
    raw = json.load(f)

data = raw if isinstance(raw, list) else list(raw.values())
print(f"Loaded {len(data)} conversations\n")

print(f"{'#':>3} {'Speakers':30} {'Sessions':>8} {'Turns':>6} {'QA':>5}")
print("─" * 58)
for i, s in enumerate(data):
    conv  = s["conversation"]
    skeys = [k for k in conv if k.startswith("session_") and "date_time" not in k]
    turns = sum(len(conv[k]) for k in skeys)
    spa   = conv.get("speaker_a", "?")
    spb   = conv.get("speaker_b", "?")
    print(f"{i:>3} {spa+' & '+spb:30} {len(skeys):>8} {turns:>6} {len(s.get('qa',[])):>5}")

Loaded 10 conversations

  # Speakers                       Sessions  Turns    QA
──────────────────────────────────────────────────────────
  0 Caroline & Melanie                   19    419   199
  1 Jon & Gina                           19    369   105
  2 John & Maria                         32    663   193
  3 Joanna & Nate                        29    629   260
  4 Tim & John                           29    680   242
  5 Audrey & Andrew                      28    675   158
  6 James & John                         31    689   190
  7 Deborah & Jolene                     30    681   239
  8 Evan & Sam                           25    509   196
  9 Calvin & Dave                        30    568   204


## 6. Data Helpers

In [6]:
def get_speakers(sample: dict) -> tuple[str, str]:
    conv = sample["conversation"]
    return conv.get("speaker_a", "?"), conv.get("speaker_b", "?")


def get_all_turns(sample: dict) -> list[dict]:
    """
    Get ALL turns across ALL sessions in chronological order.
    Each turn enriched with session metadata.
    This is the stream we process turn by turn.
    """
    conv = sample["conversation"]
    session_keys = sorted(
        [k for k in conv if k.startswith("session_") and "date_time" not in k],
        key=lambda k: int(k.split("_")[1])
    )
    all_turns = []
    for sk in session_keys:
        session_num = int(sk.split("_")[1])
        timestamp   = conv.get(f"{sk}_date_time", "")
        for turn in conv[sk]:
            if turn.get("text"):
                all_turns.append({
                    "speaker"    : turn.get("speaker", ""),
                    "text"       : turn.get("text", ""),
                    "dia_id"     : turn.get("dia_id", ""),
                    "session_key": sk,
                    "session_num": session_num,
                    "timestamp"  : timestamp,
                })
    return all_turns


def get_qa_pairs(sample: dict) -> list[dict]:
    """Get all QA pairs with evidence dia_ids and max_evidence_session."""
    result = []
    for qa in sample.get("qa", []):
        q = qa.get("question", "")
        a = qa.get("answer", "")
        if not q or not a:
            continue
        evidence = qa.get("evidence", [])
        ev_sessions = []
        for ev in evidence:
            try:
                ev_sessions.append(int(ev.split(":")[0].replace("D", "")))
            except:
                pass
        result.append({
            "question"            : q,
            "answer"              : a,
            "category"            : CATEGORY_MAP.get(qa.get("category", 0), "unknown"),
            "evidence"            : evidence,
            "max_evidence_session": max(ev_sessions) if ev_sessions else 0,
        })
    return result


def build_dia_id_to_turn(sample: dict) -> dict:
    """dia_id -> turn dict for evidence lookup."""
    lookup = {}
    for turn in get_all_turns(sample):
        lookup[turn["dia_id"]] = turn
    return lookup


# ── preview ──
sample   = data[0]
spa, spb = get_speakers(sample)
turns    = get_all_turns(sample)
qa_pairs = get_qa_pairs(sample)

print(f"Conversation 0: {spa} & {spb}")
print(f"Total turns: {len(turns)}")
print(f"Total QA: {len(qa_pairs)}")
print(f"\nFirst 3 turns:")
for t in turns[:3]:
    print(f"  {t['dia_id']:8} | {t['session_key']:10} | {t['speaker']:10} | {t['text'][:60]}")
print(f"\nFirst QA:")
q = qa_pairs[0]
print(f"  Q: {q['question']}")
print(f"  A: {q['answer']}")
print(f"  Evidence: {q['evidence']} | max_session: {q['max_evidence_session']}")

Conversation 0: Caroline & Melanie
Total turns: 419
Total QA: 154

First 3 turns:
  D1:1     | session_1  | Caroline   | Hey Mel! Good to see you! How have you been?
  D1:2     | session_1  | Melanie    | Hey Caroline! Good to see you! I'm swamped with the kids & w
  D1:3     | session_1  | Caroline   | I went to a LGBTQ support group yesterday and it was so powe

First QA:
  Q: When did Caroline go to the LGBTQ support group?
  A: 7 May 2023
  Evidence: ['D1:3'] | max_session: 1


## 7. Qwen as Memory Controller
This is what makes this different from the session pipeline.
Qwen reads each turn and decides what to store, where, and how.
Python just executes Qwen's decisions.


In [7]:
MEMORY_CONTROLLER_PROMPT = """You are a memory controller for a conversational AI system.
You are tracking information about two people: {speaker_a} and {speaker_b}.

Current dialogue turn:
{speaker}: "{text}"
(Turn ID: {dia_id} | Session: {session_key} | Date: {timestamp})

Current memory state:
{memory_context}

Your job: decide what NEW information from this turn is worth storing permanently.

Rules:
- Only store genuinely new information not already in memory
- Be specific and precise — vague statements are not worth storing
- Categorize correctly:
  FACTUAL = concrete facts (events, dates, relationships, career, identity, locations)
  PREFERENCE = likes, dislikes, values, opinions, what matters to the person
  BEHAVIORAL = communication patterns, decision-making habits, recurring behaviors
  NO_STORE = small talk, greetings, nothing new worth remembering

Output ONLY the memory operations, one per line, in this exact format:
STORE_FACT[{speaker_a}]: <specific fact about {speaker_a}>
STORE_FACT[{speaker_b}]: <specific fact about {speaker_b}>
STORE_PREFERENCE[{speaker_a}]: <preference of {speaker_a}>
STORE_PREFERENCE[{speaker_b}]: <preference of {speaker_b}>
STORE_BEHAVIOR[{speaker_a}]: <behavioral pattern of {speaker_a}>
STORE_BEHAVIOR[{speaker_b}]: <behavioral pattern of {speaker_b}>
NO_STORE

Only output lines that apply. If nothing new: output only NO_STORE."""


def parse_memory_operations(
    raw_output : str,
    speaker_a  : str,
    speaker_b  : str,
) -> list[dict]:
    """
    Parse Qwen's memory controller output into structured operations.
    Returns list of {type, speaker, content} dicts.
    """
    operations = []
    for line in raw_output.strip().split("\n"):
        line = line.strip()
        if not line or line == "NO_STORE":
            continue

        # STORE_FACT[Caroline]: content
        for op_type, mem_type in [
            ("STORE_FACT",      "factual"),
            ("STORE_PREFERENCE","preference"),
            ("STORE_BEHAVIOR",  "behavioral"),
        ]:
            if line.startswith(op_type):
                # extract speaker from brackets
                speaker = None
                bracket_match = re.search(r"\[(.+?)\]", line)
                if bracket_match:
                    speaker = bracket_match.group(1).strip()
                # extract content after colon
                colon_idx = line.find(":")
                if colon_idx >= 0:
                    content = line[colon_idx+1:].strip()
                    if content and speaker:
                        operations.append({
                            "mem_type": mem_type,
                            "speaker" : speaker,
                            "content" : f"[{speaker}] {content}",
                        })
                break

    return operations


def run_memory_controller(
    turn       : dict,
    memory     : MemorySystem,
    speaker_a  : str,
    speaker_b  : str,
    debug      : bool = DEBUG,
) -> list[dict]:
    """
    Qwen reads one turn and decides what to store.
    Returns list of operations executed.
    """
    # retrieve relevant existing memory for context
    query    = f"{turn['speaker']} {turn['text']}"
    retrieved = memory.retrieve_all(query, top_k=2)
    mem_ctx   = memory.format_for_context(retrieved)

    prompt = MEMORY_CONTROLLER_PROMPT.format(
        speaker_a    = speaker_a,
        speaker_b    = speaker_b,
        speaker      = turn["speaker"],
        text         = turn["text"],
        dia_id       = turn["dia_id"],
        session_key  = turn["session_key"],
        timestamp    = turn["timestamp"],
        memory_context = mem_ctx,
    )

    raw = qwen_generate(prompt, max_new_tokens=200)
    operations = parse_memory_operations(raw, speaker_a, speaker_b)

    if debug:
        print(f"\n    [MEMORY CONTROLLER] {turn['dia_id']} | {turn['speaker']}: {turn['text'][:60]}")
        print(f"    Qwen raw: {raw[:200]}")
        if operations:
            for op in operations:
                print(f"    → STORE_{op['mem_type'].upper()}: {op['content'][:80]}")
        else:
            print(f"    → NO_STORE")

    # execute operations
    for op in operations:
        bank = getattr(memory, op["mem_type"])
        bank.add(
            content     = op["content"],
            source_dia  = turn["dia_id"],
            session_num = turn["session_num"],
            timestamp   = turn["timestamp"],
        )

    return operations

## 8. QA Answering with Typed Memory Retrieval
When a QA pair arrives, retrieve from all three banks separately.
Each bank's results are labeled so the model knows what type each memory is.


In [8]:
QA_PHASE1_PROMPT = """You are answering a question about a long multi-session conversation.

Retrieved memory about the speakers:

{factual_memory}

{preference_memory}

{behavioral_memory}

Current context (the session where the answer is expected):
{current_context}

Question: {question}

Based on the memory and context above, reason about what information is needed.

REASONING: <2-3 sentences about where the answer is and what memory is relevant>
QUERY: <keyword query to retrieve more memory if needed>"""


QA_ANSWER_PROMPT = """Answer this question about a conversation.

Memory retrieved:
{all_memory}

Current session context:
{current_context}

Reasoning: {reasoning}

Question: {question}

Answer concisely. For dates: be specific. For lists: include all items found.
If truly not found anywhere: say "Not answerable".
Answer:"""


def answer_qa(
    qa           : dict,
    memory       : MemorySystem,
    current_turn : dict,
    sample       : dict,
    debug        : bool = DEBUG,
) -> dict:
    """
    Answer one QA pair using typed memory retrieval.
    current_turn = the turn where max_evidence_session was reached.
    """
    question     = qa["question"]
    ground_truth = qa["answer"]
    category     = qa["category"]

    # get current session context (all turns in the evidence session)
    conv    = sample["conversation"]
    ev_sess = f"session_{qa['max_evidence_session']}"
    if ev_sess in conv:
        sess_turns = conv[ev_sess]
        ctx_lines  = [f"{t.get('dia_id','')} | {t.get('speaker','')}: {t.get('text','')}"
                      for t in sess_turns if t.get("text")]
        current_context = "\n".join(ctx_lines)
    else:
        current_context = "No current context available."

    # retrieve from all three banks
    retrieved = memory.retrieve_all(question, top_k=TOP_K_PER_BANK)

    factual_ctx    = "\n".join(f"  - {e.content}" for e in retrieved["factual"])    or "None"
    preference_ctx = "\n".join(f"  - {e.content}" for e in retrieved["preference"]) or "None"
    behavioral_ctx = "\n".join(f"  - {e.content}" for e in retrieved["behavioral"]) or "None"
    all_memory_ctx = memory.format_for_context(retrieved)

    if debug:
        print(f"\n  [QA] [{category}] {question}")
        print(f"  Ground truth: {ground_truth}")
        print(f"  Evidence: {qa['evidence']}")
        print(f"  Memory state: {memory.summary()}")
        print(f"  Retrieved factual:    {len(retrieved['factual'])} entries")
        print(f"  Retrieved preference: {len(retrieved['preference'])} entries")
        print(f"  Retrieved behavioral: {len(retrieved['behavioral'])} entries")

    # Phase 1 — Qwen reasons
    phase1_prompt = QA_PHASE1_PROMPT.format(
        factual_memory    = f"[FACTUAL MEMORY]\n{factual_ctx}",
        preference_memory = f"[PREFERENCE MEMORY]\n{preference_ctx}",
        behavioral_memory = f"[BEHAVIORAL MEMORY]\n{behavioral_ctx}",
        current_context   = current_context[:1000],
        question          = question,
    )
    phase1_raw = qwen_generate(phase1_prompt, max_new_tokens=250)

    reasoning = phase1_raw
    for line in phase1_raw.split("\n"):
        if line.strip().startswith("REASONING:"):
            reasoning = line.replace("REASONING:", "").strip()
            break

    if debug:
        print(f"  Phase 1 Qwen: {phase1_raw[:200]}")
        print(f"  Reasoning: {reasoning}")

    # GPT-4o-mini answers
    answer_prompt = QA_ANSWER_PROMPT.format(
        all_memory      = all_memory_ctx,
        current_context = current_context[:800],
        reasoning       = reasoning,
        question        = question,
    )
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        max_tokens=150,
        messages=[{"role": "user", "content": answer_prompt}],
    )
    answer = response.choices[0].message.content.strip()

    if debug:
        print(f"  Answer: {answer}")
        print(f"  Ground truth: {ground_truth}")

    return {
        "question"    : question,
        "category"    : category,
        "ground_truth": ground_truth,
        "evidence"    : qa["evidence"],
        "reasoning"   : reasoning,
        "answer"      : answer,
        "mem_at_answer": memory.total(),
        "factual_retrieved"    : len(retrieved["factual"]),
        "preference_retrieved" : len(retrieved["preference"]),
        "behavioral_retrieved" : len(retrieved["behavioral"]),
    }

## 9. Judge

In [9]:
JUDGE_PROMPT = """Evaluate this answer.

Question: {question}
Ground Truth: {ground_truth}
Predicted: {predicted}

Score 0.0-1.0. Be lenient on equivalent phrasings:
- "transgender woman" = "she is transgender" → >= 0.9
- "last year" when ground truth is a year → >= 0.5
- Correct fact, different phrasing → >= 0.7
- Partial list (some items correct) → 0.4-0.6
- "Not answerable" matches not-answerable ground truth → 1.0
- Completely wrong → 0.0

Output ONLY a float. Nothing else."""


def judge(question: str, ground_truth: str, predicted: str,
          debug: bool = DEBUG) -> float:
    gt = ground_truth.lower().strip()
    pr = predicted.lower().strip()
    if ("not answerable" in gt or "not found" in gt):
        score = 1.0 if ("not answerable" in pr or "not found" in pr) else 0.0
        if debug: print(f"  Judge: {score:.2f} (adversarial)")
        return score

    resp = openai_client.chat.completions.create(
        model=OPENAI_MODEL, max_tokens=5,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            question=question, ground_truth=ground_truth, predicted=predicted,
        )}],
    )
    text  = resp.choices[0].message.content.strip()
    match = re.search(r"\d+\.?\d*", text)
    score = min(max(float(match.group()), 0.0), 1.0) if match else 0.0
    if debug: print(f"  Judge: {score:.2f}")
    return score

## 10. DEBUG — Test Each Component Before Full Run
Run this cell first to verify each component works correctly before running the full pipeline.
Tests: Qwen inference → Memory controller → Memory retrieval → QA answering → Judge


In [13]:
def run_debug(sample: dict, n_turns: int = 18, n_qa: int = 3):
    """
    Debug each component step by step.
    Processes n_turns first, then asks QA pairs whose
    max_evidence_session falls within the processed turns.
    """
    spa, spb = get_speakers(sample)
    turns    = get_all_turns(sample)
    qa_pairs = get_qa_pairs(sample)

    print("=" * 70)
    print(f"DEBUG RUN: {spa} & {spb}")
    print(f"Processing {n_turns} turns then answering eligible QA pairs")
    print("=" * 70)

    # fresh memory for debug
    debug_mem = MemorySystem("debug_user", MEMORY_DIR / "debug")

    # figure out which session we will have fully processed
    # after n_turns turns
    turns_to_process = turns[:n_turns]
    max_session_covered = max(t["session_num"] for t in turns_to_process)
    print(f"Processing turns up to session {max_session_covered}")

    # only ask QA pairs whose evidence is in sessions we have covered
    eligible_qa = [
        qa for qa in qa_pairs
        if qa["max_evidence_session"] <= max_session_covered
        and qa["max_evidence_session"] > 0  # exclude adversarial
    ][:n_qa]

    print(f"Eligible QA pairs (max_evidence_session <= {max_session_covered}): {len(eligible_qa)}")

    # ── COMPONENT 1: Memory Controller ──
    print(f"\n{'─'*70}")
    print("COMPONENT 1: MEMORY CONTROLLER")
    print("─" * 70)

    for turn in turns_to_process:
        ops = run_memory_controller(turn, debug_mem, spa, spb, debug=True)
        print(f"  Memory: {debug_mem.summary()}")

    print(f"\n  Final memory after {n_turns} turns: {debug_mem.summary()}")

    # ── COMPONENT 2: Typed Retrieval ──
    print(f"\n{'─'*70}")
    print("COMPONENT 2: TYPED MEMORY RETRIEVAL")
    print("─" * 70)

    # use actual questions from eligible QA as retrieval queries
    # this is more meaningful than hardcoded test queries
    for qa in eligible_qa[:3]:
        query = qa["question"]
        print(f"\n  Query: '{query}'")
        retrieved = debug_mem.retrieve_all(query)
        for bank_name, entries in retrieved.items():
            if entries:
                print(f"  {bank_name}:")
                for e in entries:
                    print(f"    - {e.content[:80]}")
            else:
                print(f"  {bank_name}: (empty)")

    # ── COMPONENT 3: QA Answering ──
    print(f"\n{'─'*70}")
    print("COMPONENT 3: QA ANSWERING")
    print(f"Only answering QA pairs with evidence in sessions <= {max_session_covered}")
    print("─" * 70)

    qa_results = []
    for qa in eligible_qa:
        result = answer_qa(
            qa           = qa,
            memory       = debug_mem,
            current_turn = turns[0],
            sample       = sample,
            debug        = True,
        )
        score = judge(
            question     = result["question"],
            ground_truth = str(result["ground_truth"]),
            predicted    = result["answer"],
            debug        = True,
        )
        result["judge_score"] = score
        qa_results.append(result)
        print(f"  Score: {score:.2f} | Q: {result['question'][:50]}")
        time.sleep(0.2)

    # ── COMPONENT 4: Disk Persistence ──
    print(f"\n{'─'*70}")
    print("COMPONENT 4: DISK PERSISTENCE")
    print("─" * 70)
    for bank_name in ["factual", "preference", "behavioral"]:
        path = MEMORY_DIR / "debug" / f"{bank_name}.json"
        if path.exists():
            print(f"  {bank_name}.json: {path.stat().st_size} bytes ✓")
        else:
            print(f"  {bank_name}.json: NOT FOUND ✗")

    print(f"\n{'='*70}")
    print("DEBUG COMPLETE")
    print(f"  Memory: {debug_mem.summary()}")
    print(f"  QA answered: {len(qa_results)}")
    if qa_results:
        print(f"  Mean score: {np.mean([r['judge_score'] for r in qa_results]):.3f}")
    print("=" * 70)

    return qa_results


# ── RUN DEBUG ──
debug_results = run_debug(data[0], n_turns=18, n_qa=3)

DEBUG RUN: Caroline & Melanie
Processing 18 turns then answering eligible QA pairs
Processing turns up to session 1
Eligible QA pairs (max_evidence_session <= 1): 3

──────────────────────────────────────────────────────────────────────
COMPONENT 1: MEMORY CONTROLLER
──────────────────────────────────────────────────────────────────────

    [MEMORY CONTROLLER] D1:1 | Caroline: Hey Mel! Good to see you! How have you been?
    Qwen raw: NO_STORE
    → NO_STORE
  Memory: factual=0 preference=0 behavioral=0

    [MEMORY CONTROLLER] D1:2 | Melanie: Hey Caroline! Good to see you! I'm swamped with the kids & w
    Qwen raw: NO_STORE
    → NO_STORE
  Memory: factual=0 preference=0 behavioral=0

    [MEMORY CONTROLLER] D1:3 | Caroline: I went to a LGBTQ support group yesterday and it was so powe
    Qwen raw: STORE_PREFERENCE[Caroline]: Values support groups for LGBTQ community
    → STORE_PREFERENCE: [Caroline] Values support groups for LGBTQ community
  Memory: factual=0 preference=1 behavio

## 11. Full Pipeline
Only run this after confirming debug works.
Processes all turns through memory controller, then answers all QA pairs.


In [ ]:
@dataclass
class PipelineResult:
    sample_id    : str
    speaker_a    : str
    speaker_b    : str
    turns_processed : int
    memory_ops   : list  = field(default_factory=list)
    qa_results   : list  = field(default_factory=list)

    def scores(self):
        return [r["judge_score"] for r in self.qa_results if "judge_score" in r]

    def scores_by_category(self):
        d = defaultdict(list)
        for r in self.qa_results:
            if "judge_score" in r:
                d[r["category"]].append(r["judge_score"])
        return dict(d)

    def scores_by_memory_size(self):
        """Does more memory = better scores?"""
        d = defaultdict(list)
        for r in self.qa_results:
            if "judge_score" in r:
                size = r.get("mem_at_answer", 0)
                # bin into groups
                if size <= 5:
                    key = "0-5"
                elif size <= 15:
                    key = "6-15"
                elif size <= 30:
                    key = "16-30"
                else:
                    key = "31+"
                d[key].append(r["judge_score"])
        return dict(d)


def run_full_pipeline(
    sample           : dict,
    user_id          : str = None,
    max_turns        : int = MAX_TURNS_TO_PROCESS,
    max_qa           : int = MAX_QA_TO_ANSWER,
    debug            : bool = False,  # set True for verbose output
) -> PipelineResult:
    """
    Full pipeline:
    1. Process all turns through memory controller (Qwen decides what to store)
    2. Answer QA pairs using typed memory retrieval
    """
    spa, spb = get_speakers(sample)
    turns    = get_all_turns(sample)
    qa_pairs = get_qa_pairs(sample)

    if user_id is None:
        user_id = sample.get("sample_id", "conv_0")

    memory = MemorySystem(user_id, MEMORY_DIR)

    result = PipelineResult(
        sample_id       = sample.get("sample_id", "?"),
        speaker_a       = spa,
        speaker_b       = spb,
        turns_processed = 0,
    )

    print(f"\n{'='*70}")
    print(f"FULL PIPELINE: {spa} & {spb} | {sample.get('sample_id','?')}")
    print(f"Turns: {len(turns)} | QA pairs: {len(qa_pairs)}")
    print(f"Initial memory: {memory.summary()}")
    print(f"{'='*70}")

    # ── PHASE 1: Process turns through memory controller ──
    turns_to_process = turns[:max_turns] if max_turns else turns
    print(f"\n[PHASE 1] Processing {len(turns_to_process)} turns through memory controller...")
    print(f"{'─'*70}")

    for i, turn in enumerate(tqdm(turns_to_process, desc="  Processing turns")):
        ops = run_memory_controller(turn, memory, spa, spb, debug=debug)
        result.memory_ops.extend(ops)
        result.turns_processed += 1

        # periodic status update
        if (i+1) % 20 == 0:
            print(f"  Turn {i+1}/{len(turns_to_process)} | Memory: {memory.summary()}")

    print(f"\n  Done. Memory: {memory.summary()}")
    print(f"  Total operations: {len(result.memory_ops)}")

    # ── PHASE 2: Answer QA pairs ──
    qa_to_answer = sorted(qa_pairs, key=lambda q: q["max_evidence_session"])
    if max_qa:
        qa_to_answer = qa_to_answer[:max_qa]

    print(f"\n[PHASE 2] Answering {len(qa_to_answer)} QA pairs...")
    print(f"{'─'*70}")
    print(f"{'#':>4} {'Category':12} {'Score':>6} {'Mem':>5} | Question")
    print(f"{'─'*70}")

    for i, qa in enumerate(qa_to_answer):
        mem_before = memory.total()
        try:
            qa_result = answer_qa(
                qa           = qa,
                memory       = memory,
                current_turn = turns[0],
                sample       = sample,
                debug        = debug,
            )
            score = judge(
                question     = qa_result["question"],
                ground_truth = qa_result["ground_truth"],
                predicted    = qa_result["answer"],
                debug        = debug,
            )
            qa_result["judge_score"] = score
        except Exception as e:
            print(f"  Error on Q{i+1}: {e}")
            continue

        result.qa_results.append(qa_result)
        cat = qa_result["category"]
        q   = qa_result["question"]
        print(f"{i+1:>4} {cat:12} {score:>6.2f} {mem_before:>5} | {q[:45]}")
        time.sleep(0.2)

    return result


# ── RUN ──
# Change debug=True to see every step
# Change max_turns and max_qa to limit scope
print("Starting full pipeline on conversation 0...")
print("Set debug=True to see every step (verbose)")
print("Set max_turns to limit turns processed (None = all)")
print()

pipeline_result = run_full_pipeline(
    sample    = data[0],
    user_id   = "caroline_melanie",
    max_turns = 50,    # ← process first 50 turns. set None for all 419
    max_qa    = 20,    # ← answer first 20 QA pairs
    debug     = False, # ← set True for step-by-step output
)

print(f"\nDone. {len(pipeline_result.qa_results)} questions answered.")

## 12. Results

In [ ]:
def print_results(result: PipelineResult):
    scores = result.scores()
    if not scores:
        print("No results.")
        return

    print("=" * 65)
    print(f"RESULTS: {result.speaker_a} & {result.speaker_b}")
    print("=" * 65)
    print(f"Turns processed:     {result.turns_processed}")
    print(f"Memory operations:   {len(result.memory_ops)}")
    print(f"QA pairs answered:   {len(result.qa_results)}")
    print(f"Mean score:          {np.mean(scores):.3f}")
    print(f"Pass@1 (>= 0.5):     {np.mean([s >= 0.5 for s in scores]):.1%}")

    # memory operation breakdown
    print(f"\n{'─'*65}")
    print("MEMORY OPERATIONS BREAKDOWN")
    print("─" * 65)
    from collections import Counter
    op_counts = Counter(op["mem_type"] for op in result.memory_ops)
    for mem_type, count in op_counts.items():
        print(f"  {mem_type:15}: {count} entries stored")

    # memory accumulation effect
    print(f"\n{'─'*65}")
    print("MEMORY SIZE vs SCORE (core demonstration)")
    print("─" * 65)
    by_mem = result.scores_by_memory_size()
    print(f"  {'Memory at answer':20} {'N':>5} {'Mean':>8} {'Pass@1':>8}")
    print(f"  {'─'*45}")
    for label in ["0-5", "6-15", "16-30", "31+"]:
        if label in by_mem:
            s = by_mem[label]
            print(f"  {label:20} {len(s):>5} {np.mean(s):>8.3f} {np.mean([x>=0.5 for x in s]):>8.1%}")

    # by category
    print(f"\n{'─'*65}")
    print("BY CATEGORY")
    print("─" * 65)
    by_cat = result.scores_by_category()
    print(f"  {'Category':15} {'N':>5} {'Mean':>8} {'Pass@1':>8}")
    print(f"  {'─'*40}")
    for cat in ["single-hop","multi-hop","temporal","open-domain","adversarial"]:
        if cat in by_cat:
            s = by_cat[cat]
            print(f"  {cat:15} {len(s):>5} {np.mean(s):>8.3f} {np.mean([x>=0.5 for x in s]):>8.1%}")

    # best examples where memory helped
    print(f"\n{'─'*65}")
    print("BEST EXAMPLES (high score, high memory)")
    print("─" * 65)
    best = sorted(
        [r for r in result.qa_results if "judge_score" in r],
        key=lambda r: (r["judge_score"], r.get("mem_at_answer", 0)),
        reverse=True,
    )[:5]
    for r in best:
        print(f"\n  Score: {r['judge_score']:.2f} | Mem: {r.get('mem_at_answer',0)} | [{r['category']}]")
        print(f"  Q: {r['question']}")
        print(f"  GT: {r['ground_truth']}")
        print(f"  A:  {r['answer']}")
        print(f"  Retrieved: factual={r.get('factual_retrieved',0)} "
              f"preference={r.get('preference_retrieved',0)} "
              f"behavioral={r.get('behavioral_retrieved',0)}")


print_results(pipeline_result)

## 13. Save Results

In [ ]:
def save_results(result: PipelineResult, path: str = "typed_memory_results.json"):
    output = {
        "sample_id"      : result.sample_id,
        "speakers"       : f"{result.speaker_a} & {result.speaker_b}",
        "turns_processed": result.turns_processed,
        "memory_ops"     : len(result.memory_ops),
        "mean_score"     : float(np.mean(result.scores())) if result.scores() else 0,
        "qa_results"     : result.qa_results,
    }
    with open(path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"Saved to {path}")


save_results(pipeline_result, "typed_memory_results.json")

from google.colab import files
files.download("typed_memory_results.json")